In [1]:
import pandas as pd
import numpy as np

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

df = pd.read_csv("../data/train.csv")

In [2]:
df.groupby("transaction_id")["placement_score"].nunique().describe()

count    308.000000
mean       4.412338
std        1.532103
min        2.000000
25%        3.000000
50%        4.000000
75%        5.000000
max        9.000000
Name: placement_score, dtype: float64

In [3]:
df.groupby("product_subcategory")["placement_score"].nunique().describe()

count    49.000000
mean     25.387755
std       6.254386
min       7.000000
25%      23.000000
50%      25.000000
75%      29.000000
max      37.000000
Name: placement_score, dtype: float64

In [4]:
df.groupby("transaction_id")["placement_score"].nunique().describe()

count    308.000000
mean       4.412338
std        1.532103
min        2.000000
25%        3.000000
50%        4.000000
75%        5.000000
max        9.000000
Name: placement_score, dtype: float64

In [5]:
df.groupby("product_subcategory")["placement_score"].nunique().describe()

count    49.000000
mean     25.387755
std       6.254386
min       7.000000
25%      23.000000
50%      25.000000
75%      29.000000
max      37.000000
Name: placement_score, dtype: float64

In [6]:
df[
    [
        "transaction_id",
        "product_subcategory",
        "quantity",
        "total_amount",
        "placement_score"
    ]
].head(30)

,transaction_id,product_subcategory,quantity,total_amount,placement_score
0,TXN_000001,Cheese,4,27.72,73.1
1,TXN_000001,Chicken,1,5.41,37.6
2,TXN_000001,Crackers,3,11.40,74.4
3,TXN_000001,Toothpaste,1,2.91,40.8
4,TXN_000002,Soap,1,1.87,71.9
5,TXN_000002,Shampoo,1,6.93,82.5
6,TXN_000002,Toothpaste,1,4.36,55.5
7,TXN_000002,Crackers,2,4.28,66.6
8,TXN_000002,Cheese,2,9.36,73.2
9,TXN_000003,Soap,1,1.37,67.7


In [7]:
basket_size = df.groupby("transaction_id").size()

basket_value = df.groupby("transaction_id")["total_amount"].sum()

basket_quantity = df.groupby("transaction_id")["quantity"].sum()

In [8]:
df["basket_size"] = df["transaction_id"].map(basket_size)

df["basket_value"] = df["transaction_id"].map(basket_value)

df["product_basket_share"] = (
    df["total_amount"] / df["basket_value"]
)

df["quantity_share"] = (
    df["quantity"] / df["transaction_id"].map(basket_quantity)
)

In [9]:
df[
    [
        "transaction_id",
        "product_subcategory",
        "quantity",
        "total_amount",
        "basket_size",
        "basket_value",
        "product_basket_share",
        "quantity_share",
        "placement_score"
    ]
].head(10)

,transaction_id,product_subcategory,quantity,total_amount,basket_size,basket_value,product_basket_share,quantity_share,placement_score
0,TXN_000001,Cheese,4,27.72,4,47.44,0.584317,0.444444,73.1
1,TXN_000001,Chicken,1,5.41,4,47.44,0.114039,0.111111,37.6
2,TXN_000001,Crackers,3,11.40,4,47.44,0.240304,0.333333,74.4
3,TXN_000001,Toothpaste,1,2.91,4,47.44,0.061341,0.111111,40.8
4,TXN_000002,Soap,1,1.87,5,26.80,0.069776,0.142857,71.9
5,TXN_000002,Shampoo,1,6.93,5,26.80,0.258582,0.142857,82.5
6,TXN_000002,Toothpaste,1,4.36,5,26.80,0.162687,0.142857,55.5
7,TXN_000002,Crackers,2,4.28,5,26.80,0.159701,0.285714,66.6
8,TXN_000002,Cheese,2,9.36,5,26.80,0.349254,0.285714,73.2
9,TXN_000003,Soap,1,1.37,6,51.89,0.026402,0.090909,67.7


In [10]:
product_frequency = (
    df.groupby("product_subcategory")["transaction_id"]
    .nunique()
)

product_customer_count = (
    df.groupby("product_subcategory")["customer_id"]
    .nunique()
)

product_avg_quantity = (
    df.groupby("product_subcategory")["quantity"]
    .mean()
)

product_avg_amount = (
    df.groupby("product_subcategory")["total_amount"]
    .mean()
)

In [11]:
df["product_frequency"] = (
    df["product_subcategory"].map(product_frequency)
)

df["product_customer_count"] = (
    df["product_subcategory"].map(product_customer_count)
)

df["product_avg_quantity"] = (
    df["product_subcategory"].map(product_avg_quantity)
)

df["product_avg_amount"] = (
    df["product_subcategory"].map(product_avg_amount)
)

In [12]:
df[
    [
        "product_subcategory",
        "product_frequency",
        "product_customer_count",
        "product_avg_quantity",
        "product_avg_amount"
    ]
].drop_duplicates().head(20)

,product_subcategory,product_frequency,product_customer_count,product_avg_quantity,product_avg_amount
0,Cheese,29,29,1.517241,8.996552
1,Chicken,29,28,1.586207,13.634828
2,Crackers,32,32,2.250000,7.091875
3,Toothpaste,24,23,1.625000,5.616667
4,Soap,41,38,1.926829,4.694878
5,Shampoo,40,36,2.125000,17.313500
11,Onions,38,35,1.763158,3.175526
12,Pasta,41,40,1.829268,4.297561
13,Beef,39,37,1.743590,21.579231
14,Cereals,29,27,1.827586,8.675862


In [14]:
product_max_lift = {}

for product, associations in association_features.items():
    product_max_lift[product] = max(
        a["lift"] for a in associations
    )

df["max_lift"] = (
    df["product_subcategory"]
    .map(product_max_lift)
    .fillna(0)
)

NameError: name 'association_features' is not defined

In [15]:
basket = (
    df.groupby("transaction_id")["product_subcategory"]
    .apply(list)
)

In [16]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

te = TransactionEncoder()

basket_encoded = te.fit(basket).transform(basket)

basket_df = pd.DataFrame(
    basket_encoded,
    columns=te.columns_
)

In [17]:
frequent_itemsets = apriori(
    basket_df,
    min_support=0.05,
    use_colnames=True
)

In [18]:
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0
)

In [19]:
pair_rules = rules[
    (rules["antecedents"].apply(len) == 1) &
    (rules["consequents"].apply(len) == 1)
].copy()

In [20]:
association_features = {}

for _, row in pair_rules.iterrows():
    for product in row["antecedents"]:
        association_features.setdefault(product, []).append({
            "partner": list(row["consequents"])[0],
            "lift": row["lift"],
            "confidence": row["confidence"],
            "support": row["support"]
        })

    for product in row["consequents"]:
        association_features.setdefault(product, []).append({
            "partner": list(row["antecedents"])[0],
            "lift": row["lift"],
            "confidence": row["confidence"],
            "support": row["support"]
        })

In [21]:
product_max_lift = {}

for product, associations in association_features.items():
    product_max_lift[product] = max(
        a["lift"] for a in associations
    )

df["max_lift"] = (
    df["product_subcategory"]
    .map(product_max_lift)
    .fillna(0)
)

In [22]:
product_avg_lift = {}

for product, associations in association_features.items():
    product_avg_lift[product] = np.mean(
        [a["lift"] for a in associations]
    )

df["avg_lift"] = (
    df["product_subcategory"]
    .map(product_avg_lift)
    .fillna(0)
)

In [23]:
product_max_confidence = {}

for product, associations in association_features.items():
    product_max_confidence[product] = max(
        a["confidence"] for a in associations
    )

df["max_confidence"] = (
    df["product_subcategory"]
    .map(product_max_confidence)
    .fillna(0)
)

In [24]:
product_association_count = {
    product: len(associations)
    for product, associations in association_features.items()
}

df["association_count"] = (
    df["product_subcategory"]
    .map(product_association_count)
    .fillna(0)
)

In [25]:
df[["product_category", "product_subcategory"]].drop_duplicates()

,product_category,product_subcategory
0,Dairy,Cheese
1,Meat,Chicken
2,Snacks,Crackers
3,Personal_Care,Toothpaste
4,Personal_Care,Soap
5,Personal_Care,Shampoo
11,Produce,Onions
12,Pantry,Pasta
13,Meat,Beef
14,Pantry,Cereals


In [27]:
pair_lift = {}

for _, row in pair_rules.iterrows():
    product_a = list(row["antecedents"])[0]
    product_b = list(row["consequents"])[0]

    pair = tuple(sorted([product_a, product_b]))
    pair_lift[pair] = row["lift"]

In [29]:
def get_basket_association_features(row):
    transaction_products = set(
        df.loc[
            df["transaction_id"] == row["transaction_id"],
            "product_subcategory"
        ]
    )

    current_product = row["product_subcategory"]

    lifts = []

    for other_product in transaction_products:
        if other_product == current_product:
            continue

        pair = tuple(sorted([current_product, other_product]))

        if pair in pair_lift:
            lifts.append(pair_lift[pair])

    if len(lifts) == 0:
        return pd.Series({
            "basket_partner_count": 0,
            "basket_avg_partner_lift": 0,
            "basket_max_partner_lift": 0
        })

    return pd.Series({
        "basket_partner_count": len(lifts),
        "basket_avg_partner_lift": np.mean(lifts),
        "basket_max_partner_lift": max(lifts)
    })

In [30]:
basket_association_features = df.apply(
    get_basket_association_features,
    axis=1
)

df = pd.concat(
    [df, basket_association_features],
    axis=1
)

In [31]:
df[
    [
        "basket_partner_count",
        "basket_avg_partner_lift",
        "basket_max_partner_lift"
    ]
].head(10)

,basket_partner_count,basket_avg_partner_lift,basket_max_partner_lift
0,1.0,7.965517,7.965517
1,0.0,0.000000,0.000000
2,1.0,7.965517,7.965517
3,0.0,0.000000,0.000000
4,1.0,6.197561,6.197561
5,1.0,6.197561,6.197561
6,0.0,0.000000,0.000000
7,1.0,7.965517,7.965517
8,1.0,7.965517,7.965517
9,1.0,6.197561,6.197561


In [32]:
feature_columns = [
    "quantity",
    "unit_price",
    "total_amount",
    "basket_size",
    "basket_value",
    "product_basket_share",
    "quantity_share",
    "product_frequency",
    "product_customer_count",
    "product_avg_quantity",
    "product_avg_amount",
    "max_lift",
    "avg_lift",
    "max_confidence",
    "association_count",
    "basket_partner_count",
    "basket_avg_partner_lift",
    "basket_max_partner_lift"
]

X = df[feature_columns]
y = df["placement_score"]

In [33]:
print(X.shape)
print(y.shape)

(1500, 18)
(1500,)


In [34]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [35]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=3,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_val)

In [36]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train, y_train)

gb_pred = gb_model.predict(X_val)

In [37]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_val)

In [38]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE": [
        mean_absolute_error(y_val, lr_pred),
        mean_absolute_error(y_val, rf_pred),
        mean_absolute_error(y_val, gb_pred)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_val, lr_pred)),
        np.sqrt(mean_squared_error(y_val, rf_pred)),
        np.sqrt(mean_squared_error(y_val, gb_pred))
    ],
    "R2": [
        r2_score(y_val, lr_pred),
        r2_score(y_val, rf_pred),
        r2_score(y_val, gb_pred)
    ]
})

results.sort_values("R2", ascending=False)

,Model,MAE,RMSE,R2
2,Gradient Boosting,9.033482,11.437059,0.653545
0,Linear Regression,9.199916,11.813364,0.630372
1,Random Forest,9.469897,12.000835,0.618547
